# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a guided workflow for loading and exploring the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) dataset using the `mlcroissant` library.

### Dataset Source
The dataset is defined by a Croissant schema at the following URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`


In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load the dataset metadata and records using the `mlcroissant` library.

In [ ]:
import mlcroissant as mlc
import pandas as pd
from pprint import pprint

# The Croissant schema for this dataset
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)

# Print high-level metadata
print(f"Title: {dataset.metadata.name}\n")
print(f"Description: {dataset.metadata.description}\n")
print(f"Citation: {dataset.metadata.cite_as}")

## 2. Data Overview
Review available record sets, fields, and their `@id` values.

Let's enumerate the available record sets in this dataset, and for each, list their fields and columns along with their `@id` identifiers. For all dataset exploration steps, always refer to each element by its Croissant `@id`.

In [ ]:
# List all record set @ids and their fields/columns

def list_record_sets(ds):
    record_set_objs = ds.metadata.record_sets
    if not record_set_objs:
        print("No record sets available in this dataset.")
        return []
    record_set_ids = []
    for rs in record_set_objs:
        print(f"- RecordSet @id: {rs.id}")
        print(f"  Name: {rs.name}")
        print(f"  Description: {getattr(rs, 'description', '')}")
        field_ids = []
        if hasattr(rs, 'fields') and rs.fields:
            for fld in rs.fields:
                print(f"    - Field @id: {fld.id} | Name: {fld.name} | type: {fld.data_type}")
                field_ids.append(fld.id)
        if hasattr(rs, 'columns') and rs.columns:
            for col in rs.columns:
                print(f"    - Column @id: {col.id} | Name: {col.name} | type: {col.data_type}")
        record_set_ids.append(rs.id)
    return record_set_ids

record_set_ids = list_record_sets(dataset)

## 3. Data Extraction
Extract data from each RecordSet into a DataFrame for analysis.

Use the record set and field `@id` values from the overview above to reference entities.

In [ ]:
# Extract data from every record set into a DataFrame indexed by RecordSet @id

dataframes = {}
for record_set_id in record_set_ids:
    print(f"Loading records for RecordSet: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Available columns in {record_set_id}:")
    print(df.columns.tolist())
    print(df.head(3))

# For demonstration, select the first available RecordSet by its @id
main_record_set_id = record_set_ids[0] if record_set_ids else None
if main_record_set_id:
    print(f"\nPreview of main record set ({main_record_set_id}):")
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data manipulations: filtering records, normalizing numeric fields, and grouping/categorization.

- Select a numeric field (by its `@id`) for filtering/normalization.
- Group by a relevant field (e.g., protein or sample type) if available.

**Note:** Replace `<numeric_field_id>` and `<group_field_id>` with the actual values from your RecordSet if necessary. For demonstration, we try to automatically select a likely numeric field.

In [ ]:
import numpy as np

df = dataframes.get(main_record_set_id)
if df is not None and not df.empty:
    # Try to select a numeric field by heuristic
    numeric_candidates = df.select_dtypes(include=[np.number]).columns.tolist()
    if numeric_candidates:
        numeric_field = numeric_candidates[0]  # first numeric field
        print(f"Using numeric field for filtering: {numeric_field}")
    else:
        # If no numeric detected, try string fields that could be coerced
        for col in df.columns:
            try:
                _ = pd.to_numeric(df[col])
                numeric_field = col
                print(f"Using numeric field for filtering (converted): {numeric_field}")
                df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
                break
            except Exception:
                continue
        else:
            print("No numeric field found for demonstration.")
            numeric_field = None

    if numeric_field:
        # Filter for numeric > threshold (example: 10)
        threshold = 10
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold}:")
        print(filtered_df.head())

        # Normalize the numeric field
        col_norm = f"{numeric_field}_normalized"
        filtered_df[col_norm] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"\nNormalized '{numeric_field}' for filtered records:")
        print(filtered_df[[numeric_field, col_norm]].head())

        # Attempt to group by a categorical field if present
        group_candidates = df.select_dtypes(include=[object, 'category']).columns.tolist()
        group_field = None
        for candidate in group_candidates:
            nunique = df[candidate].nunique()
            if 1 < nunique < len(df) // 2:  # Heuristic: not unique or all same
                group_field = candidate
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"\nGrouped data by '{group_field}':")
            print(grouped_df.head())
    else:
        print("No appropriate numeric field could be selected.")
else:
    print("No data available in the main record set.")

## 5. Visualization
Visualize the distribution of a numeric field, or relationship between two fields in the dataset using matplotlib or seaborn. Only plot if data is suitable.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if df is not None and numeric_field:
    plt.figure(figsize=(7, 4))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=30, color='royalblue')
    plt.title(f"Distribution of '{numeric_field}' in main record set")
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.tight_layout()
    plt.show()
    # If grouping field available, plot a boxplot
    if group_field:
        plt.figure(figsize=(8,4))
        sns.boxplot(data=df, x=group_field, y=numeric_field)
        plt.title(f"'{numeric_field}' by group: {group_field}")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

## 6. Conclusion

- The notebook demonstrates end-to-end loading and basic exploration of a Croissant-defined dataset using `mlcroissant`.
- We reviewed the available record sets and fields by their `@id`, loaded data into pandas DataFrames, applied simple EDA (filtering, normalization, grouping), and visualized distributions.
- For advanced analysis or bespoke visualizations, refer to the [mlcroissant documentation](https://mlcommons.org/croissant/) and schema definitions from this dataset.
